In [ ]:
import os
import shutil
from pathlib import Path

# Define workspace directory paths
WORKING_DIR = Path("/kaggle/working")
REPO_NAME = "clickbait-spoiling-IR-final-project"
REPO_DIR = WORKING_DIR / REPO_NAME
REPO_URL = f"https://github.com/aryonmt/{REPO_NAME}.git"


def setup_workspace():
    """Clones the repository and configures the current active working directory."""
    # Clean up existing repository directory to ensure a fresh, idempotent run
    if REPO_DIR.exists():
        print(f"Cleaning existing directory: {REPO_DIR}")
        shutil.rmtree(REPO_DIR)

    # Clone the repository directly to the Kaggle working workspace
    print(f"Cloning repository from {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

    # Verify the directory structure is fully resolved
    if REPO_DIR.exists():
        print(f"Workspace resolved successfully. Changing directory to {REPO_DIR}")
        os.chdir(REPO_DIR)
    else:
        raise FileNotFoundError(f"Failed to clone repository into {REPO_DIR}")


setup_workspace()

In [ ]:
from pathlib import Path

# Kaggle dataset mount. The slug below is the dataset host, separate from the GitHub owner aryonmt.
# Define Kaggle input source paths
PRIMARY_TRAIN_SOURCE = Path(
    "/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/train.jsonl"
)
PRIMARY_VAL_SOURCE = Path(
    "/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/validation.jsonl"
)

FALLBACK_TRAIN_SOURCE = Path("/kaggle/input/click-bait-spoiling-dataset/train.jsonl")
FALLBACK_VAL_SOURCE = Path("/kaggle/input/click-bait-spoiling-dataset/validation.jsonl")

PROJECT_DATA_DIR = Path("./data")


def align_datasets():
    """Links external Kaggle dataset splits to the project's internal data directory."""
    # Ensure local project data folder exists
    PROJECT_DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_source = PRIMARY_TRAIN_SOURCE
    val_source = PRIMARY_VAL_SOURCE

    # Fallback resolution check in case Kaggle mount directory shifts
    if not train_source.exists() or not val_source.exists():
        print("[WARNING] Primary dataset path not found. Checking fallback path...")
        train_source = FALLBACK_TRAIN_SOURCE
        val_source = FALLBACK_VAL_SOURCE

    if not train_source.exists() or not val_source.exists():
        raise FileNotFoundError(
            "Could not resolve dataset paths in Kaggle inputs directory."
        )

    train_target = PROJECT_DATA_DIR / "train.jsonl"
    val_target = PROJECT_DATA_DIR / "validation.jsonl"

    # Clean up existing symbolic links or stale files to maintain idempotency
    if train_target.is_symlink() or train_target.exists():
        train_target.unlink()
    if val_target.is_symlink() or val_target.exists():
        val_target.unlink()

    # Generate symbolic links
    train_target.symlink_to(train_source)
    val_target.symlink_to(val_source)

    print("Dataset successfully mapped to project workspace:")
    print(f" - Train Link: {train_target.resolve()} -> {train_target.readlink()}")
    print(f" - Val Link: {val_target.resolve()} -> {val_target.readlink()}")


align_datasets()

In [ ]:
import os
import importlib

# Define key packages to verify post-installation
REQUIRED_PACKAGES = ["transformers", "datasets", "torch", "evaluate", "sentencepiece"]


def install_dependencies():
    """Installs required packages from the project's requirements.txt file."""
    print("Installing requirements from requirements.txt...")
    exit_code = os.system("pip install -r requirements.txt --quiet")
    if exit_code != 0:
        raise RuntimeError("Failed to install required dependencies.")
    print("[SUCCESS] Dependencies installed.")


def verify_installations():
    """Imports and prints installed versions of core project dependencies."""
    print("\nVerifying installed package versions:")
    for package in REQUIRED_PACKAGES:
        try:
            module = importlib.import_module(package)
            version = getattr(module, "__version__", "unknown")
            print(f" - {package}: {version}")
        except ImportError:
            print(f" - [ERROR] {package} is not installed.")


install_dependencies()
verify_installations()

In [ ]:
import torch


def verify_gpu_acceleration():
    """Checks PyTorch CUDA support and displays specifications of the active GPU."""
    cuda_active = torch.cuda.is_available()
    print("=" * 60)
    print(f"CUDA GPU Acceleration Active: {cuda_active}")
    print("=" * 60)

    if cuda_active:
        device_name = torch.cuda.get_device_name(0)
        allocated_memory = torch.cuda.memory_allocated(0) / (1024**2)
        print(f"Device Name: {device_name}")
        print(f"Allocated VRAM: {allocated_memory:.2f} MB")
    else:
        print(
            "[WARNING] Running on CPU mode. Transformer training will be extremely slow."
        )
        print(
            "Go to Notebook settings (right panel) -> Accelerator -> Select 'T4 GPU'."
        )


verify_gpu_acceleration()

In [ ]:
import os


def run_eda_analysis():
    """Runs the main exploratory data analysis script on the Kaggle train split."""
    print("Launching Exploratory Data Analysis (EDA)...")
    exit_code = os.system("python main_eda.py")
    if exit_code != 0:
        raise RuntimeError("EDA processing failed.")

    print("[SUCCESS] EDA analysis complete. Generated artifacts in reports directory:")
    # Verify the generated plots in the output directory
    os.system("ls -la reports/")


run_eda_analysis()

In [ ]:
import os


def train_task1_classifier():
    """Fine-tunes the Task 1 sequence classifier using the defaults configured in config.py."""
    print("Starting Task 1 Sequence Classifier training...")

    # We let main_transformer_task1.py automatically read model_name, lr,
    # output_dir, and epochs from the updated PipelineConfig in config.py.
    cmd = (
        "python main_transformer_task1.py "
        "--train_path data/train.jsonl "
        "--val_path data/validation.jsonl"
    )

    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Task 1 Sequence Classifier training pipeline failed.")

    print("[SUCCESS] Task 1 sequence classification fine-tuning completed safely.")


train_task1_classifier()

In [ ]:
import os


def train_phrase_qa_model():
    """Fine-tunes the extractive QA model on phrase-type spoilers using config.py defaults."""
    print("Starting Task 2 Phrase QA model training...")
    cmd = (
        "python main_transformer_task2_train.py "
        "--spoiler_type phrase "
        "--train_path data/train.jsonl "
        "--val_path data/validation.jsonl"
    )
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Phrase QA model fine-tuning failed.")
    print("[SUCCESS] Phrase QA model fine-tuning completed successfully.")


train_phrase_qa_model()

In [ ]:
import os


def train_passage_qa_model():
    """Fine-tunes the extractive QA model on passage-type spoilers using config.py defaults."""
    print("Starting Task 2 Passage QA model training...")
    cmd = (
        "python main_transformer_task2_train.py "
        "--spoiler_type passage "
        "--train_path data/train.jsonl "
        "--val_path data/validation.jsonl"
    )
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Passage QA model fine-tuning failed.")
    print("[SUCCESS] Passage QA model fine-tuning completed successfully.")


train_passage_qa_model()

In [ ]:
import os


def train_multi_seq2seq_model():
    """Fine-tunes the generative T5 Seq2Seq model for resolving multi-sentence spoilers

    using the dataset splits and defaults configured in config.py.
    """
    print("Starting Task 2 Generative Multi Seq2Seq model training (Flan-T5)...")

    # We let main_transformer_task2_multi_seq2seq.py automatically read
    # model_name (google/flan-t5-base) and default lr (5e-5) from the CLI defaults.
    cmd = (
        "python main_transformer_task2_multi_seq2seq.py "
        "--train_path data/train.jsonl "
        "--val_path data/validation.jsonl"
    )

    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Generative Multi Seq2Seq model fine-tuning failed.")

    print(
        "[SUCCESS] Generative Multi Seq2Seq T5 model training completed successfully."
    )


train_multi_seq2seq_model()

In [ ]:
import os


def run_integrated_pipeline():
    """Orchestrates end-to-end inference reading model checkpoints and parameters

    directly from the default PipelineConfig (including the trained T5 weights).
    """
    print("Launching end-to-end Integrated Pipeline Inference with Generative T5...")

    # All directories (including results_multi_seq2seq) and strategy (seq2seq)
    # are automatically resolved from your config.py file.
    cmd = (
        "python main_integrated_pipeline.py "
        "--val_path data/validation.jsonl "
        "--t1_checkpoint outputs "
        "--phrase_checkpoint results_qa_phrase "
        "--passage_checkpoint results_qa_passage"
    )

    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Integrated pipeline inference failed.")

    print("[SUCCESS] Final submission 'final_integrated_submission.jsonl' generated.")


run_integrated_pipeline()

In [ ]:
import os


def generate_html_report():
    """Generates visual charts and the interactive evaluation HTML report."""
    print("Generating final visual evaluation reports...")
    cmd = (
        "python main_generate_report.py "
        "--predictions_path final_integrated_submission.jsonl"
        " --val_path data/validation.jsonl"
        " --output_dir reports"
    )
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Visual report generation failed.")
    print("[SUCCESS] Visualization reports compiled.")


generate_html_report()

In [ ]:
import shutil
from pathlib import Path

# Define the source directory to compress and the target archive file name
REPORTS_DIR = Path("reports")
ZIP_OUTPUT_NAME = "clickbait_spoiling_evaluation_reports"


def archive_reports_directory():
    """Compresses the entire reports folder into a single ZIP archive for download."""
    if REPORTS_DIR.exists():
        print(f"Compressing all files inside '{REPORTS_DIR}'...")
        # Generates 'clickbait_spoiling_evaluation_reports.zip' in the active workspace
        archive_path = shutil.make_archive(ZIP_OUTPUT_NAME, "zip", REPORTS_DIR)
        print(f"[SUCCESS] ZIP archive successfully compiled at: {archive_path}")
    else:
        print(
            f"[ERROR] Directory '{REPORTS_DIR}' not found. Verify report generation ran successfully."
        )


archive_reports_directory()